In [1]:
import pandas as pd
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

/home/an00b/Anup2026/Software/omnidata-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


def run_nepali_evaluation(model_name="BAAI/bge-m3", k=3):
    print(f"--- Starting Nepali Evaluation for Model: {model_name} ---")
    
    # 1. Load the Corpus (company_ne.csv)
    print("Loading company_ne.csv...")
    corpus_df = pd.read_csv('company_ne.csv')
    
    # Clean any NA values
    corpus_df = corpus_df.dropna(subset=['text', 'id', 'section'])
    
    doc_ids = corpus_df['id'].tolist()
    
    # Combine the section heading and the actual text for better context mapping
    # Example: "३. कम्पनीको संस्थापना (१) मुनाफाको उद्देश्य लिई..."
    doc_texts = (corpus_df['section'] + " \n " + corpus_df['text']).tolist()
    
    # 2. Load the Nepali Evaluation Queries
    print("Loading eval_queries_ne.json...")
    with open('eval_queries_ne.json', 'r', encoding='utf-8') as f:
        eval_data = json.load(f)
    
    queries = [item['query'] for item in eval_data]
    expected_ids = [item['expected_id'] for item in eval_data]

    # 3. Initialize Model and Generate Embeddings
    print(f"Loading embedding model '{model_name}' (this may take a moment)...")
    model = SentenceTransformer(model_name)
    
    print(f"Embedding {len(doc_texts)} Nepali document chunks...")
    corpus_embeddings = model.encode(doc_texts, show_progress_bar=True, convert_to_numpy=True)
    
    print(f"Embedding {len(queries)} test queries...")
    query_embeddings = model.encode(queries, show_progress_bar=True, convert_to_numpy=True)
    
    # 4. Calculate Cosine Similarity Matrix
    print("Calculating similarity and metrics...")
    similarity_matrix = cosine_similarity(query_embeddings, corpus_embeddings)
    
    # 5. Evaluate Metrics (MRR and Hit Rate)
    hit_count = 0
    reciprocal_ranks = []
    
    for idx, (query_text, expected_id) in enumerate(zip(queries, expected_ids)):
        query_scores = similarity_matrix[idx]
        
        # Sort indices highest to lowest similarity
        sorted_indices = np.argsort(query_scores)[::-1]
        ranked_doc_ids = [doc_ids[i] for i in sorted_indices]
        
        # Top-K for Hit Rate
        top_k_results = ranked_doc_ids[:k]
        is_hit = expected_id in top_k_results
        if is_hit:
            hit_count += 1
            
        # MRR Calculation
        try:
            rank = ranked_doc_ids.index(expected_id) + 1
            reciprocal_ranks.append(1.0 / rank)
        except ValueError:
            reciprocal_ranks.append(0.0)
            
        print(f"\nQuery: {query_text}")
        print(f"Expected ID: {expected_id} | Retrieved Top-{k}: {top_k_results} | Hit: {is_hit}")

    # 6. Final Results
    final_hit_rate = hit_count / len(queries)
    final_mrr = np.mean(reciprocal_ranks)
    
    print("\n" + "="*50)
    print(f"FINAL METRICS FOR {model_name} (NEPALI DATASET)")
    print("="*50)
    print(f"Hit Rate @ {k}: {final_hit_rate * 100:.2f}%")
    print(f"Mean Reciprocal Rank (MRR): {final_mrr:.4f}")
    print("="*50)



In [3]:
if __name__ == "__main__":
    # Test with bge-m3, you can later test "intfloat/multilingual-e5-large"
    run_nepali_evaluation(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", k=3)

--- Starting Nepali Evaluation for Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 ---
Loading company_ne.csv...
Loading eval_queries_ne.json...
Loading embedding model 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' (this may take a moment)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7834.67it/s]


Embedding 165 Nepali document chunks...


Batches: 100%|██████████| 6/6 [00:00<00:00,  9.70it/s]


Embedding 4 test queries...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.72it/s]

Calculating similarity and metrics...

Query: कम्पनी ऐन, २०६३ कहिलेदेखि प्रारम्भ भएको मानिनेछ?
Expected ID: company_01_001 | Retrieved Top-3: ['company_01_001', 'company_05_003', 'company_16_003'] | Hit: True

Query: पब्लिक कम्पनी संस्थापना गर्न कम्तीमा कति जना संस्थापक हुनुपर्छ?
Expected ID: company_02_001 | Retrieved Top-3: ['company_02_005', 'company_01_001', 'company_21_009'] | Hit: False

Query: विदेशी कम्पनी भनेको के हो?
Expected ID: company_01_002 | Retrieved Top-3: ['company_16_002', 'company_21_004', 'company_01_002'] | Hit: True

Query: कम्पनीको नाम स्वीकृतिको लागि कार्यालय समक्ष कसरी निवेदन दिनु पर्छ?
Expected ID: company_02_002 | Retrieved Top-3: ['company_21_009', 'company_02_005', 'company_03_009'] | Hit: False

FINAL METRICS FOR sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 (NEPALI DATASET)
Hit Rate @ 3: 50.00%
Mean Reciprocal Rank (MRR): 0.3403
